# SOCOFing robustness study — full reproducible run (A + B)

Self-contained Kaggle notebook. Re-run it top-to-bottom any time:

- **Part A — Experiments:** every model × level × condition. Cached gallery
  descriptors + per-probe resume make re-runs cheap (only missing work is done).
- **Part B — Analysis:** tables, figures, and PAIRED SIGNIFICANCE (McNemar +
  bootstrap), then bundles everything for download.

**Before running:** attach the SOCOFing dataset, and enable **Internet**
(Settings → Internet ON) so DINOv2 weights download via torch.hub.

## Setup

In [ ]:
import os
if not os.path.exists('/kaggle/working/Biometric'):
    !git clone https://github.com/AndreaGiurissich/Biometric.git /kaggle/working/Biometric
%cd /kaggle/working/Biometric
!git pull origin main
!pip install -q -r requirements.txt

In [ ]:
# Confirm the dataset is mounted and parsed as expected.
!python scripts/verify_dataset.py --input-root /kaggle/input

## Part A — Experiments

All 3 models × 3 levels × 2 conditions (Tier A). SIFT is O(N×gallery) so its
scoring is parallelized with `--workers 4`. `--no-synthesize --no-significance`
keeps this cell to pure experiments; Part B does the analysis. Safe to re-run:
finished probes are skipped, gallery descriptors are cached.

In [ ]:
!python scripts/run_all.py --workers 4 --no-synthesize --no-significance

## Part B — Analysis

Tables + figures, then the inferential layer (is the preprocessing effect
significant, per model × level?).

In [ ]:
!python scripts/synthesize.py        # results/summary.csv, per_alteration.csv, figures/
!python scripts/significance.py      # results/significance.csv (McNemar + bootstrap)

In [ ]:
# Figures
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/results/figures/*.png')):
    print(p); display(Image(p))

In [ ]:
# Tables
import pandas as pd
print('== overall =='); display(pd.read_csv('/kaggle/working/results/summary.csv'))
print('== per alteration =='); display(pd.read_csv('/kaggle/working/results/per_alteration.csv'))
print('== significance =='); display(pd.read_csv('/kaggle/working/results/significance.csv'))

In [ ]:
# Bundle results for download (Kaggle disk is ephemeral -> save BEFORE the session ends).
!python scripts/save_results.py

### Optional — Tier B (best estimate, full probe set)

Gabor + DINOv2 on every probe (not the 2000-subsample). Heavier; SIFT stays Tier
A. Run only if you want the full-data numbers, then re-run Part B.

```python
!python scripts/run_all.py --models gabor,dinov2 --full --workers 4 \
    --no-synthesize --no-significance
```